In [4]:
import sys
if 'google.colab' in sys.modules:
    from IPython.core.getipython import get_ipython
    get_ipython().run_line_magic("pip", "install transformers sentencepiece accelerate")
    get_ipython().run_line_magic("pip", "install git+https://github.com/UlisseMini/activation_additions_hf")

In [48]:
import torch
import math
import activation_additions as aa

from typing import Dict, Union, Callable
from functools import partial
from transformers import AutoModelForCausalLM, AutoTokenizer
from activation_additions.compat import get_x_vector
from functools import lru_cache

import nltk
from nltk.tokenize import PunktTokenizer
from tqdm.notebook import tqdm

from numpy import array,polyfit

In [6]:
device: str = "mps" if torch.has_mps else "cuda" if torch.cuda.is_available() else "cpu"
_ = torch.set_grad_enabled(False)
device

C:\Users\itayb\AppData\Local\Temp\ipykernel_79596\1209224586.py:1: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  device: str = "mps" if torch.has_mps else "cuda" if torch.cuda.is_available() else "cpu"


'cuda'

In [50]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\itayb\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [8]:
MODEL = "openai-community/gpt2-xl"
model = AutoModelForCausalLM.from_pretrained(MODEL).to(device)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model.to_str_tokens = lambda t: [t.replace('Ġ', ' ') for t in tokenizer.tokenize(t)]
model.tokenizer = tokenizer
# In steering experimentation spaces were found to work well, this makes no sense and I hate it.
tokenizer.pad_token_id = int(model.tokenizer.encode(" ")[-1])

Loading weights:   0%|          | 0/580 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2-xl
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...47}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
sampling_kwargs: Dict[str, Union[float, int]] = {
    "temperature": 0.6,
    "top_p": 0.5,
    "freq_penalty": 1.0,
    "num_comparisons": 1,
    "tokens_to_generate": 50,
    "seed": 0,  # For reproducibility
}
get_x_vector_preset: Callable = partial(
    get_x_vector,
    pad_method="tokens_right",
    model=model,
    custom_pad_id=tokenizer.pad_token_id,
)

In [10]:
@lru_cache(maxsize=1000)
def get_diff_vector(prompt_add: str, prompt_sub: str, layer: int):
    return aa.get_diff_vector(model, tokenizer, prompt_add, prompt_sub, layer)


@lru_cache
def run_aa(coeff: float, layer: int, prompt_add: str, prompt_sub: str, texts: tuple[str], loss_ignore_mod_tokens: bool = False):
    # todo: could compute act_diff for all layers at once, then a single fwd pass of cost for changing layer.
    act_diff = coeff * get_diff_vector(prompt_add, prompt_sub, layer)
    blocks = aa.get_blocks(model)
    with aa.pre_hooks([(blocks[layer], aa.get_hook_fn(act_diff))]):
        inputs = tokenizer(list(texts), return_tensors='pt', padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        output = model(**inputs,output_hidden_states=True, return_dict=True)

    logprobs = torch.log_softmax(output.logits.to(torch.float32), -1)
    token_loss = -logprobs[..., :-1, :].gather(dim=-1, index=inputs['input_ids'][..., 1:, None])[..., 0]
    return token_loss

In [ ]:
def calculate_perplexity_change(args_obj):
    sent_detector = PunktTokenizer()
    texts = args_obj['texts']
    loss_diff_in_bin = dict()
    element_count_in_bin = dict()
    for text in texts:
        prompt_add_len,prompt_sub_len = args_obj["prompt_add"],args_obj["prompt_sub"]
        sentences = tuple(sent for sent in sent_detector.tokenize(text.strip()) if (len(sent) >= prompt_add_len) and len(sent) >= len(prompt_sub_len))
        mean_text_log_loss = 0   
        token_loss = run_aa(args_obj["coeff"],args_obj["layer"],args_obj["prompt_add"],args_obj["prompt_sub"],sentences,args_obj["loss_ignore_mod_tokens"])
        abs_token_loss = run_aa(0., 0, ' ', ' ', texts=sentences) # cached
        mean_text_log_loss = sum(sent.mean() for sent in token_loss)/len(sentences)
        abs_mean_text_log_loss = sum(sent.mean() for sent in abs_token_loss)/len(sentences)

        rel_bin = texts[text]
        loss_diff_in_bin[rel_bin] = loss_diff_in_bin.get(rel_bin,0) + (mean_text_log_loss - abs_mean_text_log_loss)
        element_count_in_bin[rel_bin] = element_count_in_bin.get(rel_bin,0) + 1
    
    x = []
    y = []
    for bin in loss_diff_in_bin:
        perplexity_diff_in_bin = math.exp(-loss_diff_in_bin[bin])
        x.append(bin)
        y.append(perplexity_diff_in_bin)

    #print(list(zip(x,y)))
    slope,_ = polyfit(x,y,1)
    return slope

In [12]:
def get_prompt_add(concept: str) -> str:
    return "I constantly talk about weddings"
def get_prompt_sub(concept: str) -> str:
    return "I never talk about weddings"
def get_test_texts(concept: str) -> dict[str,float]:
    return {
        "The weather is wonderful today.\nI wonder how well the game's going.":0,
        "Class today was interesting, but nothing noteworthy.\nThe teacher is quite repetitive.": 0,
        "Are you buying the gift for the groom today?\nI hope the bride wont mind.":1,
        "The bachelor party already happened\nI hope they live happily together ":1,
        "I am so excited about the wedding\nI can't wait to get married!":3,
        "The wedding rehersal starts in less than 30 minutes\nWe don't want to keep the bride and groom waiting.":3
    }

In [23]:
def calculate_steering_efficacy_over_concept(concept, coeff, layer):
    summand_dict = {
        "coeff":coeff,
        "layer":layer,
        "prompt_add":get_prompt_add(concept),
        "prompt_sub":get_prompt_sub(concept),
        "texts": get_test_texts(concept),
        "loss_ignore_mod_tokens":False
    }
    return calculate_perplexity_change(summand_dict)
calculate_steering_efficacy_over_concept("weddings",6,8)

np.float64(0.04703797865460286)

In [53]:
from numpy import random
def P(E,E_next,temp):
    prob = math.exp(-(E_next-E)/temp)
    if (prob > 1): return True
    else: return random.binomial(1,prob)

def random_neighbor(coeff,layer,coeff_delta = 0.1):
    coeff_opts = [0]
    if coeff > coeff_delta: coeff_opts.append(-coeff_delta)
    if coeff < 15: coeff_opts.append(coeff_delta)
    coeff_d = random.choice(coeff_opts)
    layer_opts = []
    if coeff_d != 0: layer_opts.append(0)
    if layer > 0: layer_opts.append(-1)
    if layer < model.config.num_hidden_layers-1: layer_opts.append(1)
    layer_d = random.choice(layer_opts)
    return coeff + coeff_d,layer + layer_d

    

def basic_hyperparamter_optimization(concept,k_max=100, init_temp = 10, decay_rate = 0.99):
    coeff,layer = random.uniform(0.01,8),random.randint(0,model.config.num_hidden_layers//2)
    E = calculate_steering_efficacy_over_concept(concept,coeff,layer)
    print("Starting:",coeff,layer,E)
    T = init_temp
    best_E = (E,(coeff,layer))
    iterable = range(round(math.log(1/init_temp,decay_rate)))
    pbar = tqdm()
    pbar.reset(total=len(iterable))
    while T > 1:
        pbar.update()
        #print(T)
        neighbor = random_neighbor(coeff,layer)
        E_next = calculate_steering_efficacy_over_concept(concept,coeff,layer)
        #print(coeff,layer,E_next)
        if P(E,E_next,T):
            coeff,layer = neighbor
            if best_E[0] > E_next:
                best_E = (E_next,(coeff,layer))
            E = E_next
        T *= decay_rate
    return best_E



In [54]:
basic_hyperparamter_optimization("test")

Starting: 0.329134524310701 18 0.02738261789540175


0it [00:00, ?it/s]

(np.float64(-0.008847651926358426),
 (np.float64(0.129134524310701), np.int64(26)))